# Benchmark Copula-MSM VaR: Baseline vs Optimized

This notebook compares the initial code and optimized versions on a small number of rolling windows to avoid full 500 forecasts.

Methodology:
- same rolling scheme (fixed window),
- same MSM and copula estimation,
- same VaR equation.

Tested modes:
- baseline (reference),
- optimized exact (equivalent methodology),
- optimized interp (quantile approximation).

Secondly, we try different parameters estimations in order to compare the execution time to the precision of the results.

Tested estimations:
- fixed parameters,
- parameters estimation every x days,
- parameters estimation triggered by likelihood level.

In [2]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.var import (
    forecast_msm_copula_var_rolling,
    forecast_msm_copula_var_rolling_optimized,
    forecast_msm_copula_var_models_parallel,
)

def timed_call(label, fn, **kwargs):
    t0 = perf_counter()
    out = fn(**kwargs)
    dt = perf_counter() - t0
    print(f"{label}: {dt:.2f}s")
    return out, dt

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
returns_path = PROJECT_ROOT / "data" / "processed" / "returns_nasdaq_sp500.csv"

df = pd.read_csv(returns_path)

date_col_candidates = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
if date_col_candidates:
    date_col = date_col_candidates[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(date_col)

ret = df.select_dtypes(include=["number"]).dropna(how="any")
if ret.shape[1] != 2:
    raise ValueError(f"Expected exactly 2 return columns, got {ret.shape[1]}: {list(ret.columns)}")

ret = ret.iloc[:, :2]
ret.head(), ret.shape

(              NASDAQ   S&P 500
 date                          
 2009-04-16  2.647210  1.541931
 2009-04-17  0.157320  0.495705
 2009-04-20 -3.953850 -4.373221
 2009-04-21  2.191930  2.102938
 2009-04-22  0.137996 -0.771132,
 (1635, 2))

In [ ]:
# Parametres benchmark: small n_oos to limit time
copula = "student"
alpha = 0.05
weights = (0.5, 0.5)
k = 5
n_insample = 1135
n_oos_benchmark = 10   # increase progressively: 10 -> 20 -> 50
n_starts = 3           # fast benchmark; increase for more robust results
integration_nodes = 201
root_tol = 1e-4

print({
    "copula": copula,
    "alpha": alpha,
    "k": k,
    "n_insample": n_insample,
    "n_oos_benchmark": n_oos_benchmark,
    "n_starts": n_starts,
    "integration_nodes": integration_nodes,
})

{'copula': 'student', 'alpha': 0.05, 'k': 5, 'n_insample': 1135, 'n_oos_benchmark': 10, 'n_starts': 3, 'integration_nodes': 201}


In [ ]:
# Baseline (reference)
baseline, t_baseline = timed_call(
    "baseline",
    forecast_msm_copula_var_rolling,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_benchmark,
    n_starts=n_starts,
    integration_nodes=integration_nodes,
    root_tol=root_tol,
    verbose=False,
)
baseline.head()

baseline: 620.98s


2013-10-17   -1.636370
2013-10-18   -1.575935
2013-10-21   -1.604065
2013-10-22   -1.490101
2013-10-23   -1.401817
Name: CopulaMSM_student_VaR_0.05, dtype: float64

In [ ]:
# Optimized strictly equivalent (quantile_mode='exact')
opt_exact, t_opt_exact = timed_call(
    "optimized_exact",
    forecast_msm_copula_var_rolling_optimized,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_benchmark,
    n_starts=n_starts,
    integration_nodes=integration_nodes,
    root_tol=root_tol,
    quantile_mode="exact",
    verbose=False,
)
opt_exact.head()

optimized_exact: 358.50s


2013-10-17   -1.636370
2013-10-18   -1.575935
2013-10-21   -1.604065
2013-10-22   -1.490101
2013-10-23   -1.401817
Name: CopulaMSMOptimized_student_VaR_0.05, dtype: float64

In [ ]:
# Optimized approximation (quantile_mode='interp')
opt_interp, t_opt_interp = timed_call(
    "optimized_interp",
    forecast_msm_copula_var_rolling_optimized,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_benchmark,
    n_starts=n_starts,
    integration_nodes=integration_nodes,
    root_tol=root_tol,
    quantile_mode="interp",
    quantile_grid_size=4096,
    verbose=False,
)
opt_interp.head()

optimized_interp: 1487.69s


2013-10-17   -1.636393
2013-10-18   -1.575966
2013-10-21   -1.604095
2013-10-22   -1.490092
2013-10-23   -1.401844
Name: CopulaMSMOptimized_student_VaR_0.05, dtype: float64

In [10]:
cmp = pd.concat([
    baseline.rename("baseline"),
    opt_exact.rename("optimized_exact"),
    opt_interp.rename("optimized_interp"),
], axis=1).dropna()

err_exact = (cmp["optimized_exact"] - cmp["baseline"]).abs()
err_interp = (cmp["optimized_interp"] - cmp["baseline"]).abs()

summary = pd.DataFrame({
    "runtime_sec": [t_baseline, t_opt_exact, t_opt_interp],
    "speedup_vs_baseline": [1.0, t_baseline / t_opt_exact, t_baseline / t_opt_interp],
    "mae_vs_baseline": [0.0, err_exact.mean(), err_interp.mean()],
    "max_abs_err_vs_baseline": [0.0, err_exact.max(), err_interp.max()],
}, index=["baseline", "optimized_exact", "optimized_interp"])

summary

,runtime_sec,speedup_vs_baseline,mae_vs_baseline,max_abs_err_vs_baseline
baseline,620.983347,1.000000,0.00000,0.000000
optimized_exact,358.504811,1.732148,0.00000,0.000000
optimized_interp,1487.686452,0.417415,0.00003,0.000054


In [11]:
# Extrapolation lineaire indicative vers 500 predictions (sans relancer complet)
target_n_oos = 500
scale = target_n_oos / n_oos_benchmark

proj = pd.DataFrame({
    "measured_sec": [t_baseline, t_opt_exact, t_opt_interp],
    "projected_sec_for_500": [t_baseline * scale, t_opt_exact * scale, t_opt_interp * scale],
    "projected_hours_for_500": [(t_baseline * scale) / 3600.0, (t_opt_exact * scale) / 3600.0, (t_opt_interp * scale) / 3600.0],
}, index=["baseline", "optimized_exact", "optimized_interp"])

proj

,measured_sec,projected_sec_for_500,projected_hours_for_500
baseline,620.983347,31049.167365,8.624769
optimized_exact,358.504811,17925.240535,4.979233
optimized_interp,1487.686452,74384.322625,20.662312


In [ ]:
# Parallel estimation for several copulas
copulas_small = ["student", "gaussian"]
panel_parallel, t_parallel = timed_call(
    "parallel_panel",
    forecast_msm_copula_var_models_parallel,
    returns=ret,
    copulas=copulas_small,
    n_jobs=2,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=min(8, n_oos_benchmark),
    n_starts=n_starts,
    integration_nodes=integration_nodes,
    quantile_mode="interp",
    verbose=False,
)
panel_parallel.head()

parallel_panel: 642.52s


,Copula-MSM student,Copula-MSM gaussian
2013-10-17,-1.636393,-1.636897
2013-10-18,-1.575966,-1.576478
2013-10-21,-1.604095,-1.604660
2013-10-22,-1.490092,-1.490698
2013-10-23,-1.401844,-1.402387


## Extended benchmark: fixed parameters, periodic refit, and likelihood-triggered refit

This section adds three low-cost strategies for the Copula-MSM VaR forecast:

- fixed parameters from the first estimation,
- periodic re-estimation every `x` rolling windows,
- re-estimation only when the likelihood becomes too poor.

The goal is to compare these strategies against the original full refit baseline without launching the full 500-window experiment.

In [15]:
from src.copulas import (
    copula_loglikelihood,
    fit_copula,
)
from src.msm import (
    fit_msm,
    make_msm_states,
    msm_filter_from_result,
    msm_loglikelihood,
    msm_mixture_cdf,
    msm_mixture_quantile,
    renewal_probabilities_from_gamma_k,
    transition_matrix_from_gammas,
)
from src.var import (
    prepare_bivariate_returns,
    rolling_windows,
    solve_portfolio_var_fast,
)


def msm_inverse_exact(u, state_probs, sigma, h, mean, root_tol):
    u_arr = np.asarray(u, dtype=float)
    u_flat = np.clip(u_arr.reshape(-1), 1e-10, 1.0 - 1e-10)
    values = [
        msm_mixture_quantile(
            float(ui),
            state_probs=state_probs,
            sigma=sigma,
            h=h,
            mean=mean,
            root_tol=root_tol,
        )
        for ui in u_flat
    ]
    out = np.asarray(values, dtype=float).reshape(u_arr.shape)
    if np.ndim(u) == 0:
        return float(out)
    return out


def fit_copula_msm_window(window, copula, k, n_starts, seed):
    assets = list(window.columns)
    r1 = window[assets[0]]
    r2 = window[assets[1]]

    msm_1 = fit_msm(
        returns=r1,
        k=k,
        n_starts=n_starts,
        seed=seed + 1,
        verbose=False,
    )
    msm_2 = fit_msm(
        returns=r2,
        k=k,
        n_starts=n_starts,
        seed=seed + 2,
        verbose=False,
    )

    filt_1 = msm_filter_from_result(r1, msm_1)
    filt_2 = msm_filter_from_result(r2, msm_2)

    pit = pd.concat(
        [
            filt_1["pit"].rename(assets[0]),
            filt_2["pit"].rename(assets[1]),
        ],
        axis=1,
    ).dropna()

    cop_fit = fit_copula(pit, copula=copula, margin_model="MSM")

    gammas_1 = renewal_probabilities_from_gamma_k(
        k=k,
        b=msm_1.params.b,
        gamma_k=msm_1.params.gamma_k,
    )
    gammas_2 = renewal_probabilities_from_gamma_k(
        k=k,
        b=msm_2.params.b,
        gamma_k=msm_2.params.gamma_k,
    )

    A_1 = transition_matrix_from_gammas(gammas_1)
    A_2 = transition_matrix_from_gammas(gammas_2)

    h1 = np.sqrt(np.prod(make_msm_states(k=k, m0=msm_1.params.m0), axis=1))
    h2 = np.sqrt(np.prod(make_msm_states(k=k, m0=msm_2.params.m0), axis=1))

    return {
        "assets": assets,
        "msm_1": msm_1,
        "msm_2": msm_2,
        "cop_fit": cop_fit,
        "A_1": A_1,
        "A_2": A_2,
        "h1": h1,
        "h2": h2,
    }


def forecast_from_model(window, model, alpha, weights, root_tol, integration_nodes):
    assets = model["assets"]
    r1 = window[assets[0]]
    r2 = window[assets[1]]

    filt_1 = msm_filter_from_result(r1, model["msm_1"])
    filt_2 = msm_filter_from_result(r2, model["msm_2"])

    p1 = filt_1["filtered_probs"].iloc[-1].to_numpy(dtype=float) @ model["A_1"]
    p2 = filt_2["filtered_probs"].iloc[-1].to_numpy(dtype=float) @ model["A_2"]

    def inv1(u):
        return msm_inverse_exact(
            u=u,
            state_probs=p1,
            sigma=model["msm_1"].params.sigma,
            h=model["h1"],
            mean=model["msm_1"].mean_return,
            root_tol=root_tol,
        )

    def inv2(u):
        return msm_inverse_exact(
            u=u,
            state_probs=p2,
            sigma=model["msm_2"].params.sigma,
            h=model["h2"],
            mean=model["msm_2"].mean_return,
            root_tol=root_tol,
        )

    def cdf1(x):
        return msm_mixture_cdf(
            x=x,
            state_probs=p1,
            sigma=model["msm_1"].params.sigma,
            h=model["h1"],
            mean=model["msm_1"].mean_return,
        )

    return solve_portfolio_var_fast(
        alpha=alpha,
        inverse_cdf_1=inv1,
        inverse_cdf_2=inv2,
        cdf_1=cdf1,
        copula_params=model["cop_fit"].params,
        copula=model["cop_fit"].copula,
        pi=float(weights[0]),
        integration_nodes=integration_nodes,
        root_tol=root_tol,
    )


def model_average_loglik(window, model):
    assets = model["assets"]
    r1 = window[assets[0]]
    r2 = window[assets[1]]

    ll1 = msm_loglikelihood(
        y=r1 - model["msm_1"].mean_return,
        k=model["msm_1"].k,
        m0=model["msm_1"].params.m0,
        sigma=model["msm_1"].params.sigma,
        b=model["msm_1"].params.b,
        gamma_k=model["msm_1"].params.gamma_k,
    )
    ll2 = msm_loglikelihood(
        y=r2 - model["msm_2"].mean_return,
        k=model["msm_2"].k,
        m0=model["msm_2"].params.m0,
        sigma=model["msm_2"].params.sigma,
        b=model["msm_2"].params.b,
        gamma_k=model["msm_2"].params.gamma_k,
    )

    filt_1 = msm_filter_from_result(r1, model["msm_1"])
    filt_2 = msm_filter_from_result(r2, model["msm_2"])
    pit = pd.concat(
        [
            filt_1["pit"].rename(assets[0]),
            filt_2["pit"].rename(assets[1]),
        ],
        axis=1,
    ).dropna()

    ll_cop = copula_loglikelihood(pit, model["cop_fit"].copula, model["cop_fit"].params)
    return float((ll1 + ll2 + ll_cop) / len(window))


def forecast_msm_copula_var_fixed_params(
    returns,
    copula,
    alpha,
    weights,
    k,
    n_insample,
    n_oos,
    n_starts,
    seed,
    root_tol,
    integration_nodes,
):
    frame = prepare_bivariate_returns(returns, n_insample=n_insample, n_oos=n_oos)
    initial_window = frame.iloc[:n_insample]
    model = fit_copula_msm_window(initial_window, copula, k, n_starts, seed)

    values = []
    dates = []
    for _, date, window in rolling_windows(frame, n_insample, n_oos):
        values.append(forecast_from_model(window, model, alpha, weights, root_tol, integration_nodes))
        dates.append(date)

    return pd.Series(values, index=dates, name=f"CopulaMSM_fixed_{copula}_VaR_{alpha:g}")


def forecast_msm_copula_var_periodic_refit(
    returns,
    copula,
    alpha,
    weights,
    k,
    n_insample,
    n_oos,
    n_starts,
    seed,
    root_tol,
    integration_nodes,
    reestimate_every=3,
):
    frame = prepare_bivariate_returns(returns, n_insample=n_insample, n_oos=n_oos)
    model = fit_copula_msm_window(frame.iloc[:n_insample], copula, k, n_starts, seed)
    values = []
    dates = []

    for i, date, window in rolling_windows(frame, n_insample, n_oos):
        if i > 0 and i % reestimate_every == 0:
            model = fit_copula_msm_window(window, copula, k, n_starts, seed + 1000 * i)
        values.append(forecast_from_model(window, model, alpha, weights, root_tol, integration_nodes))
        dates.append(date)

    return pd.Series(values, index=dates, name=f"CopulaMSM_periodic_{reestimate_every}_{copula}_VaR_{alpha:g}")


def forecast_msm_copula_var_likelihood_trigger(
    returns,
    copula,
    alpha,
    weights,
    k,
    n_insample,
    n_oos,
    n_starts,
    seed,
    root_tol,
    integration_nodes,
    ll_threshold=-4.5,
):
    frame = prepare_bivariate_returns(returns, n_insample=n_insample, n_oos=n_oos)
    model = fit_copula_msm_window(frame.iloc[:n_insample], copula, k, n_starts, seed)
    values = []
    dates = []
    refits = 1

    for i, date, window in rolling_windows(frame, n_insample, n_oos):
        if i > 0:
            avg_ll = model_average_loglik(window, model)
            if not np.isfinite(avg_ll) or avg_ll < ll_threshold:
                model = fit_copula_msm_window(window, copula, k, n_starts, seed + 1000 * i)
                refits += 1
        values.append(forecast_from_model(window, model, alpha, weights, root_tol, integration_nodes))
        dates.append(date)

    series = pd.Series(values, index=dates, name=f"CopulaMSM_trigger_{copula}_VaR_{alpha:g}")
    series.attrs["refits"] = refits
    return series

In [ ]:
# New strategy benchmark on the same short rolling sample
n_oos_compare = n_oos_benchmark
reestimate_every = 3
ll_threshold = -4.5

fixed_params, t_fixed_params = timed_call(
    "fixed_params",
    forecast_msm_copula_var_fixed_params,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_compare,
    n_starts=n_starts,
    root_tol=root_tol,
    integration_nodes=integration_nodes,
)

periodic_refit, t_periodic_refit = timed_call(
    "periodic_refit",
    forecast_msm_copula_var_periodic_refit,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_compare,
    n_starts=n_starts,
    root_tol=root_tol,
    integration_nodes=integration_nodes,
    reestimate_every=reestimate_every,
)

likelihood_trigger, t_likelihood_trigger = timed_call(
    "likelihood_trigger",
    forecast_msm_copula_var_likelihood_trigger,
    returns=ret,
    copula=copula,
    alpha=alpha,
    weights=weights,
    k=k,
    n_insample=n_insample,
    n_oos=n_oos_compare,
    n_starts=n_starts,
    root_tol=root_tol,
    integration_nodes=integration_nodes,
    ll_threshold=ll_threshold,
)

likelihood_trigger_refits = likelihood_trigger.attrs.get("refits", np.nan)

comparison = pd.concat(
    [
        baseline.rename("baseline"),
        fixed_params.rename("fixed_params"),
        periodic_refit.rename(f"periodic_{reestimate_every}"),
        likelihood_trigger.rename("likelihood_trigger"),
        opt_exact.rename("optimized_exact"),
        opt_interp.rename("optimized_interp"),
    ],
    axis=1,
).dropna()

comparison_errors = pd.DataFrame(
    {
        "mae_vs_baseline": (comparison.sub(comparison["baseline"], axis=0).abs().mean()),
        "max_abs_err_vs_baseline": (comparison.sub(comparison["baseline"], axis=0).abs().max()),
    }
)

comparison_times = pd.Series(
    {
        "baseline": t_baseline,
        "fixed_params": t_fixed_params,
        f"periodic_{reestimate_every}": t_periodic_refit,
        "likelihood_trigger": t_likelihood_trigger,
        "optimized_exact": t_opt_exact,
        "optimized_interp": t_opt_interp,
    },
    name="runtime_sec",
)

comparison_summary = pd.concat(
    [
        comparison_times,
        comparison_times / t_baseline,
        comparison_errors["mae_vs_baseline"],
        comparison_errors["max_abs_err_vs_baseline"],
    ],
    axis=1,
)
comparison_summary.columns = [
    "runtime_sec",
    "runtime_vs_baseline",
    "mae_vs_baseline",
    "max_abs_err_vs_baseline",
]
comparison_summary.loc[:, "runtime_vs_baseline"] = comparison_summary["runtime_vs_baseline"].astype(float)
comparison_summary

print(f"Likelihood-trigger refits: {likelihood_trigger_refits}")

fixed_params: 36.79s
periodic_refit: 365.54s
likelihood_trigger: 34.81s
Likelihood-trigger refits: 1


In [17]:
comparison_summary

,runtime_sec,runtime_vs_baseline,mae_vs_baseline,max_abs_err_vs_baseline
baseline,620.983347,1.000000,0.000000,0.000000
fixed_params,36.785370,0.059237,0.010043,0.015355
periodic_3,365.543806,0.588653,0.008104,0.029327
likelihood_trigger,34.806496,0.056051,0.010043,0.015355
optimized_exact,358.504811,0.577318,0.000000,0.000000
optimized_interp,1487.686452,2.395695,0.000030,0.000054
